In [ ]:
# =====================================================================
# Summary_Figures_Colab.py  -  paste into ONE Colab cell and run it.
# Draws the figures for the "Summary of Results" section from the result files of
# Final_Analysis.py stored in Google Drive. It only reads and plots; nothing is calculated.
# Output (PNG) is saved in the same Drive folder as the CSV files:
#   Figure_S1_godspeed_by_condition.png, Figure_S2_sensors_by_condition.png,
#   Figure_S3_reported_35_correlations.png, Figure_S4_heart_vs_camera.png
# =====================================================================
import os, glob
from google.colab import drive
drive.mount("/content/drive")

# Option A: write the folder that holds 04_correlations_pooled.csv, e.g.
#   "/content/drive/MyDrive/Thesis/final_results"
# Option B: leave it empty and the cell searches your Drive for the folder.
DRIVE_FOLDER = ""

if not DRIVE_FOLDER:
    hits = glob.glob("/content/drive/MyDrive/**/04_correlations_pooled.csv", recursive=True)
    hits = sorted(hits, key=os.path.getmtime, reverse=True)
    if not hits:
        raise FileNotFoundError("04_correlations_pooled.csv not found in your Drive. Set DRIVE_FOLDER.")
    print("Found these result folders (newest first):")
    for h in hits: print("  ", os.path.dirname(h))
    DRIVE_FOLDER = os.path.dirname(hits[0])
print("Using:", DRIVE_FOLDER)
need = ["02_descriptives_by_condition.csv", "04_correlations_pooled.csv",
        "06_godspeed_by_condition.csv", "10_heart_vs_camera_correlations.csv"]
missing = [f for f in need if not os.path.exists(os.path.join(DRIVE_FOLDER, f))]
assert not missing, f"Missing in {DRIVE_FOLDER}: {missing}"

import sys, os
import numpy as np, pandas as pd
import matplotlib
import matplotlib.pyplot as plt

folder = DRIVE_FOLDER
rd = lambda f: pd.read_csv(os.path.join(folder, f))
ds, gd, co, hc = rd("02_descriptives_by_condition.csv"), rd("06_godspeed_by_condition.csv"), rd("04_correlations_pooled.csv"), rd("10_heart_vs_camera_correlations.csv")
out = lambda f: os.path.join(folder, f)
plt.rcParams.update({"font.family": "serif", "font.serif": ["Times New Roman", "Liberation Serif", "DejaVu Serif"], "font.size": 10})

CONDS = ["Robot1", "Robot2", "Robot3", "Robot4", "Human"]
GS = ["anthropomorphism", "animacy", "likeability", "perceived_intelligence", "perceived_safety"]
GS_NAME = {"anthropomorphism": "Anthropomorphism", "animacy": "Animacy", "likeability": "Likeability",
           "perceived_intelligence": "Perceived intelligence", "perceived_safety": "Perceived safety"}
GS_LAB = ["Anthropo-\nmorphism", "Animacy", "Likeability", "Perceived\nintelligence", "Perceived\nsafety"]
COL = ["#1f4e79", "#c0504d", "#4f8a3c", "#8064a2", "#e69f00"]
x = np.arange(len(CONDS))

def top_label(ax, text, dx=0.0):
    """vertical-axis name written horizontally at the TOP of the vertical axis"""
    ax.text(0, 1.02, text, transform=ax.transAxes, ha="center", va="bottom", fontsize=9)

# ---------- S1: Godspeed ratings per video ----------
g = gd[gd["table"] == "descriptives"]
fig, ax = plt.subplots(figsize=(7.2, 4.4), dpi=300)
for gs, c in zip(GS, COL):
    s = g[g.godspeed_subscale == gs].set_index("condition").reindex(CONDS)
    ax.plot(x, s["mean"], "-o", color=c, lw=2.2, label=GS_NAME[gs])
ax.set_xticks(x); ax.set_xticklabels(CONDS); ax.set_ylim(1, 5)
top_label(ax, "Mean rating (1 to 5)", -0.0); ax.set_xlabel("Video, in the order shown (machine-like to human)")
ax.grid(axis="y", color="#dddddd"); ax.set_axisbelow(True)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=3, fontsize=8, frameon=False)
ax.set_title("Godspeed ratings for each video", pad=24)
fig.tight_layout(); fig.savefig(out("Figure_S1_godspeed_by_condition.png")); plt.close(fig)

# ---------- S2: every sensor measure per video (mean +/- SD) ----------
ds = ds[ds["tier"] == "primary"]          # exploratory AUs are not discussed in the thesis
meas = list(dict.fromkeys(ds["measure"]))
labs = {m: ds[ds.measure == m]["label"].iloc[0] for m in meas}
ncol = 4; nrow = int(np.ceil(len(meas) / ncol))
fig, axs = plt.subplots(nrow, ncol, figsize=(9.5, 2.6 * nrow), dpi=300, squeeze=False)
for ax, m in zip(axs.flat, meas):
    s = ds[ds.measure == m].set_index("condition").reindex(CONDS)
    ax.errorbar(x, s["mean"], yerr=s["sd"], fmt="-o", color="#1f4e79", ecolor="#9db7d1", capsize=3, lw=1.8, ms=4)
    top_label(ax, labs[m] if m == "rmssd_delta" else labs[m] + " (mean value)"); ax.set_xticks(x); ax.set_xticklabels(["R1", "R2", "R3", "R4", "H"], fontsize=8)
    ax.grid(axis="y", color="#e5e5e5"); ax.set_axisbelow(True)
    if m == "rmssd_delta": ax.axhline(0, color="grey", lw=0.8)
for ax in list(axs.flat)[len(meas):]: ax.axis("off")
fig.suptitle("Sensor measures for each video (mean and SD across participants)", fontsize=11)
fig.text(0.5, 0.005, "R1 to R4 = Robot1 to Robot4, H = Human", ha="center", fontsize=8)
fig.tight_layout(rect=(0, 0.03, 1, 0.95)); fig.savefig(out("Figure_S2_sensors_by_condition.png")); plt.close(fig)

# ---------- S3: the 35 reported sensor x Godspeed correlations (7 primary measures) ----------
co = co[co["tier"] == "primary"]
order = [m for m in meas if m in set(co["measure"])]
R = np.full((len(order), len(GS)), np.nan); P = R.copy(); H = R.copy()
for i, m in enumerate(order):
    for j, gs in enumerate(GS):
        row = co[(co.measure == m) & (co.godspeed_subscale == gs)]
        if len(row):
            R[i, j] = row.pearson_r.iloc[0]; P[i, j] = row.pearson_p.iloc[0]; H[i, j] = row.pearson_p_holm_all_55.iloc[0]
fig, ax = plt.subplots(figsize=(7.4, 4.9), dpi=300)
im = ax.imshow(R, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
for i in range(R.shape[0]):
    for j in range(R.shape[1]):
        if np.isnan(R[i, j]): continue
        st = "*" if P[i, j] < .05 else ""
        ax.text(j, i, f"{R[i, j]:.2f}{st}", ha="center", va="center", fontsize=9,
                fontweight="bold" if st else "normal", color="white" if abs(R[i, j]) > .6 else "black")
ax.set_xticks(range(len(GS))); ax.set_xticklabels(GS_LAB)
ax.set_yticks(range(len(order))); ax.set_yticklabels([labs[m] for m in order])
ax.set_xlabel("Godspeed subscale"); top_label(ax, "Sensor measure", -0.0)
cb = fig.colorbar(im, ax=ax, fraction=0.04); cb.ax.set_title("Pearson r", fontsize=9, loc="center")
ax.set_title("The 35 reported correlations: sensor measures and Godspeed ratings", pad=24)
fig.text(0.01, 0.01, f"* p < .05 (unadjusted): {int((P < .05).sum())} of {P.size}. Significant after Holm correction (adjusted for all 55 tests): {int((H < .05).sum())} of {P.size}.", fontsize=8)
fig.tight_layout(rect=(0, 0.03, 1, 1)); fig.savefig(out("Figure_S3_reported_35_correlations.png")); plt.close(fig)

# ---------- S4: all heart x camera correlations ----------
h = hc.sort_values("pearson_r").reset_index(drop=True)
fig, ax = plt.subplots(figsize=(7.2, 4.4), dpi=300)
y = np.arange(len(h))
ax.hlines(y, h.ci95_low, h.ci95_high, color="#9db7d1", lw=2.5)
ax.plot(h.pearson_r, y, "o", color="#1f4e79")
ax.axvline(0, color="grey", lw=1)
ax.set_yticks(y); ax.set_yticklabels(h["label"]); ax.set_xlim(-1, 1)
ax.set_xlabel("Pearson r with the change in RMSSD (point = r, line = 95% interval)")
top_label(ax, "Camera measure"); ax.set_title(f"All {len(h)} correlations: heart measure and camera measures", pad=24)
fig.text(0.01, 0.01, f"p < .05 (unadjusted): {int((h.pearson_p < .05).sum())} of {len(h)}. After Holm: {int(h.significant_after_holm.sum())} of {len(h)}.", fontsize=8)
fig.tight_layout(rect=(0, 0.03, 1, 1)); fig.savefig(out("Figure_S4_heart_vs_camera.png")); plt.close(fig)
print("saved 4 figures in", folder)